# RAG and Agent Evaluation

Video: [Watch this lesson](https://www.youtube.com/watch?v=VKHBP0QSCFo&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

So far, we evaluated retrieval. We checked whether search returns the
document that should answer the question.

That is only the first step. A complete application still needs to
produce a final answer. For RAG, this means checking the generated
answer. For agents, it also means looking at the tool calls the model
made before producing the answer.

RAG evaluation checks the whole flow together.

This includes:

- search
- prompt
- LLM

If the final answer is bad, the problem can come from any of these
steps. The search might retrieve the wrong document, the prompt might
omit important context, or the LLM might ignore the context.

In this part, we'll evaluate:

- RAG answers with an LLM judge
- Agent answers and tool-call trajectories

We won't go deep into agent evaluation frameworks here. We'll use the
agent from module 01, save the final answer, and also save the tool
calls. Then we can look at whether the answer is good and whether the
trajectory looks reasonable.

## LLM as a judge

For RAG and agent evaluation, we compare the generated answer with the
original answer. The generated answer won't use the same words as the
original. It's a generative model, so the phrasing will be different
even when the meaning is the same.

This is why we use another LLM to do the comparison. We show the judge
the question, the original answer, and the generated answer. Then we ask
it to decide if they are semantically equivalent.

This approach is called LLM-as-a-judge. The evaluating LLM is the
judge. It classifies each answer as good or bad and explains its
reasoning. Asking the judge to explain why it made a decision generally
produces better classifications than asking for just the verdict.

Next, we'll start with the RAG case and generate answers for the ground
truth questions.

> Get the data (it's not gonna use but it could be useful)

```python
%store -r relevance_total
```

```python
%store -r relevance_total_text
```

# Generating RAG Answers

In the first part of this module, we evaluated search quality. We
checked whether the right document appeared in the search results.

Now we evaluate the full RAG pipeline. For each generated question, we
run RAG and save the answer produced by the LLM. Later, we'll compare
this answer with the original FAQ answer.

This is the A->Q->A' setup:

- A = original answer in the FAQ
- Q = generated question from this answer
- A' = answer produced by our RAG system

If A' is close to A, the RAG system is doing a good job.

This is still offline evaluation. We can compare A and A' because our
questions came from FAQ records. For each question, we know which
original answer it came from.

Video: [Watch this lesson](https://www.youtube.com/watch?v=utkcclfpj0g&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)


## Loading the data

Create a new notebook for RAG evaluation.

Load the ground truth questions:

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
ground_truth[10]

{'question': 'How do students join the Office Hours or live workshop sessions if the Zoom link isn’t public?',
 'document': '489dd1c9d9'}

Load the FAQ documents and the search index:


In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

Create a lookup table for the original FAQ documents:


In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [5]:
q = ground_truth[10]
q

{'question': 'How do students join the Office Hours or live workshop sessions if the Zoom link isn’t public?',
 'document': '489dd1c9d9'}

In [6]:
doc_idx[q["document"]]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

## Running RAG

Import the usual things first:

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

For this lesson, use `RAGWithUsage` from the evaluation utilities. It
subclasses `RAGBase` from module 01, so it has the same `rag` method.

It stores token usage after each LLM call. Then we can calculate the
total cost later.

It also uses the search boosts we selected in the search tuning lesson:
`question=1.0`, `answer=2.0`, and `section=0.1`.

In [8]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

For each question, `RAGBase` searches the FAQ, builds a prompt with the
retrieved context, and asks the LLM to answer. We save the answer so the
next lesson can judge it.

Run RAG for one question:

In [9]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join and follow along. The course materials and videos are available, and you can start whenever you want.\n\nIf you want a certificate, though, you need to submit your project while submissions are still being accepted, and certificates are only available if you finish with a live cohort.'

Check the cost of this call:


In [10]:
assistant.total_cost()

0.000753

Get the original answer from the document ID:



In [11]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

Now save both answers in one record:


In [12]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'I just found this course late — can I still join and follow along?',
 'answer_llm': 'Yes, you can still join and follow along. The course materials and videos are available, and you can start whenever you want.\n\nIf you want a certificate, though, you need to submit your project while submissions are still being accepted, and certificates are only available if you finish with a live cohort.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

## Processing all questions

Create a function that processes one ground truth record:

In [13]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

Test it on one record:

In [14]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'I just found this course late — can I still join and follow along?',
 'answer_llm': 'Yes, you can still join and follow along.\n\nYou can start whenever you want, and the course videos and GitHub materials are available. If you want a certificate, though, you need to submit your project while submissions are still open and finish with the live cohort.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [15]:
record = generate_rag_answer(q)
record

{'question': 'How do students join the Office Hours or live workshop sessions if the Zoom link isn’t public?',
 'answer_llm': 'Students join via **YouTube Live** rather than the public Zoom link.\n\n- The **Zoom link is only for instructors/presenters/TAs**\n- The **live video URL is posted in the announcements channel on Telegram and Slack** before the session starts\n- You can also watch on the **DataTalksClub YouTube Channel**\n- Questions should be submitted via **Slido**, not chat, since chat messages may be missed',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be mi

Before running the full batch, reset the usage we collected while
testing:

In [ ]:
assistant.total_cost()

0.002502

In [17]:
assistant.reset_usage()

In [18]:
assistant.total_cost()

0.0

This calls the LLM once per ground truth question, so it can take some
time. Let's process the questions in parallel and track progress.

Import the parallel processing helper from the same utility file:

In [19]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

Run RAG for all ground truth questions:    

In [20]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/515 [00:00<?, ?it/s]

In [21]:
results[:10]

[{'question': 'I just found this course late — can I still join and follow along?',
  'answer_llm': 'Yes, you can still join and follow along. The course materials are available, and you can start whenever you want.\n\nIf you want a certificate, though, you must submit your project while submissions are still being accepted and finish with a live cohort.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I start the course after it begins, do I still have a chance to get a certificate?',
  'answer_llm': 'Yes — if you join after the course has started, you can still get a certificate as long as you submit your project while submissions are still being accepted and complete the required capstone peer-reviews during the live course.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accep

In [22]:
df_results = pd.DataFrame(results)

In [35]:
df_results.head()

,question,answer_llm,answer_orig,document
0,I just found this course late — can I still jo...,"Yes, you can still join and follow along. The ...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,"If I start the course after it begins, do I st...",Yes — if you join after the course has started...,"Yes, but if you want to receive a certificate,...",74eb249bbf
2,Is it okay to enroll now even though I missed ...,Yes — you can start whenever you want. The vid...,"Yes, but if you want to receive a certificate,...",74eb249bbf
3,What do I need to do to be eligible for the co...,"If you join late, you can still get the certif...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,"Can I still take part in the course now, or is...","Yes, you can still join the course now. If you...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [24]:
assistant.total_cost()

0.5327985000000008

In [25]:
df_results.to_csv("../data/rag-results-new.csv", index=False)

`generate_rag_answer` returns one answer record for each question.

Collect the answer records:

In [26]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [27]:
answers[:10]

[{'question': 'I just found this course late — can I still join and follow along?',
  'answer_llm': 'Yes, you can still join and follow along. The course materials are available, and you can start whenever you want.\n\nIf you want a certificate, though, you must submit your project while submissions are still being accepted and finish with a live cohort.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I start the course after it begins, do I still have a chance to get a certificate?',
  'answer_llm': 'Yes — if you join after the course has started, you can still get a certificate as long as you submit your project while submissions are still being accepted and complete the required capstone peer-reviews during the live course.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accep

Save the answers:


In [28]:
df_answers = pd.DataFrame(answers)

In [30]:
df_answers.to_csv("../data/rag-answers-new.csv", index=False)

In [34]:
df_answers.head()

,question,answer_llm,answer_orig,document
0,I just found this course late — can I still jo...,"Yes, you can still join and follow along. The ...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,"If I start the course after it begins, do I st...",Yes — if you join after the course has started...,"Yes, but if you want to receive a certificate,...",74eb249bbf
2,Is it okay to enroll now even though I missed ...,Yes — you can start whenever you want. The vid...,"Yes, but if you want to receive a certificate,...",74eb249bbf
3,What do I need to do to be eligible for the co...,"If you join late, you can still get the certif...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,"Can I still take part in the course now, or is...","Yes, you can still join the course now. If you...","Yes, but if you want to receive a certificate,...",74eb249bbf


Calculate the total cost:

In [37]:
assistant.total_cost()

0.5327985000000008

We generated this file for the course materials on May 29, 2026. The
run used 395 ground truth questions.

The total cost was $0.34332825, about 34 cents.

If you don't want to generate the RAG answers yourself, download the
file we prepared:

```bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main

wget -O data/rag-answers-new.csv ${PREFIX}/04-evaluation/data/rag-answers-new.csv
```

In the next lesson, we'll evaluate these answers with an LLM judge.